# 電流値を変化させた時の励磁磁場分布の調査  
作成日：2026年09月02日  
作成者：上杉 健太

In [ ]:
# conf
project_name = "260911_歪-励磁磁場_キャンセルコイルなし"

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

file_pattern = f"source/{project_name}/*.csv"
csv_files = sorted(glob.glob(file_pattern, recursive=True))

plt.figure(figsize=(15, 8))

target_x = 'z'
target_y = 'mf.normB (T)'
plotted_count = 0

# 指定範囲の背景色を設定 (データ線の下に表示されるよう先に描画)
plt.axvspan(-7, -1, color='orange', alpha=0.3)
plt.axvspan(1, 7, color='orange', alpha=0.3)
plt.axvspan(-1, 1, color='skyblue', alpha=0.3)

for file_path in csv_files:
    if not os.path.isfile(file_path):
        continue

    filename = os.path.basename(file_path)
    try:
        df = pd.read_csv(file_path, comment='%')
        df.columns = df.columns.str.strip()

        if target_x not in df.columns or target_y not in df.columns:
            df = pd.read_csv(file_path, comment='%', sep=r'\s+')
            df.columns = df.columns.str.strip()

        if target_x in df.columns and target_y in df.columns:
            df = df.sort_values(by=target_x, ascending=True)
            plt.plot(df[target_x], df[target_y], label=filename)
            plotted_count += 1
        else:
            print(f"スキップ: {filename} (検出された列: {list(df.columns)})")

    except Exception as e:
        print(f"エラー発生 ({filename}): {e}")

if plotted_count > 0:
    plt.xlabel(target_x)
    plt.ylabel(target_y)
    plt.title("Magnetic Flux Density Distribution of the STPG370 Pipe When the Current Value Is Varying")
    
    plt.xlim(-250, 250)
    plt.ylim(0, 2.2)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()
    plt.savefig(f"./results/{project_name}/comsol_comparison.png", dpi=300)
    plt.show()
    print(f"処理完了: {plotted_count} 件のファイルをプロットしました。")
else:
    print("有効なデータが見つかりませんでした。")